In [1]:
!pip install tensorflow tensorflow-hub

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
import sentencepiece

from tqdm.auto import tqdm

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, f1_score, classification_report


/usr/local/lib/python3.13/dist-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


### **1. Carregamento dos dados**

In [29]:
# train = pd.read_csv("train.csv")
# train = pd.read_csv("train_noDuplicates.csv")
train = pd.read_csv("train_noSameClassDuplicates.csv")

print(train.shape)
train.head()

X = train["resp_text"]
y = train["clarity"]

print("Número de textos:", len(X))
print("Número de rótulos:", len(y))

(18813, 3)
Número de textos: 18813
Número de rótulos: 18813


### **2. Divisão do conjunto em treino e validação/teste**

In [30]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=100
)

print("Treino:", X_train.shape)
print("Validação:", X_val.shape)

print("\nClasses no treino:")
print(y_train.value_counts())

print("\nClasses na validação:")
print(y_val.value_counts())

Treino: (15050,)
Validação: (3763,)

Classes no treino:
clarity
c5      5278
c234    5170
c1      4602
Name: count, dtype: int64

Classes na validação:
clarity
c5      1320
c234    1292
c1      1151
Name: count, dtype: int64


### **3. Carregamento do modelo USE**

In [5]:
import tensorflow_text as text

print("TensorFlow:", tf.__version__)
print("TensorFlow Hub:", hub.__version__)
print("TensorFlow Text:", text.__version__)

TensorFlow: 2.20.0
TensorFlow Hub: 0.16.1
TensorFlow Text: 2.20.1


In [6]:
model_use = hub.load(
    "https://tfhub.dev/google/universal-sentence-encoder-multilingual/3"
)

print("USE carregado!")

USE carregado!


In [31]:
texto_exemplo = X_train.iloc[0]
print(texto_exemplo)

embedding_exemplo = model_use([texto_exemplo])
print("Tipo:", type(embedding_exemplo))
print("Shape:", embedding_exemplo.shape)
print(embedding_exemplo[0][:10])

 Prezada Fernanda,   O Instituto Federal agradece o seu contato!   Recebemos seu pedido de informação sob nº [23480005632201928]. Informamos que sua demanda fora encaminhada para a Diretoria de Gestão de Pessoas, que posteriormente direcionou o pedido ao Departamento de Admissão, Dimensionamento e Movimentação de Pessoal do Instituto Federal do Maranhão. Diante disso, apresentamos a resposta do referido Departamento: Informamos que a vaga do Campus Codó foi ofertada em processo de redistribuição. Ressaltamos que a redistribuição é o deslocamento de cargo de provimento efetivo, ocupado ou vago no âmbito do quadro geral de pessoal, para outro órgão ou entidade do mesmo Poder, com prévia apreciação do órgão central do SIPEC (Ministério da Economia) que dentre seus requisitos destaca-se exclusivo interesse da administração. Vale ressaltar que naquele momento não havia concurso em vigência para o Instituto Federal do Maranhão e que o Campus Codó dispõe de autonomia para gerenciamento dos se

### **4. Função para gerar todos os embeddings em lotes**

In [32]:
def gerar_embeddings_use(textos, model, batch_size=32):
    embeddings = []

    textos = list(textos)

    for i in tqdm(
        range(0, len(textos), batch_size),
        desc="Gerando embeddings USE"
    ):
        batch = textos[i:i + batch_size]

        batch_embeddings = model(batch).numpy()

        embeddings.append(batch_embeddings)

    return np.vstack(embeddings)

### **5. Geração dos embeddings do treino**

In [33]:
X_train_embeddings_use = gerar_embeddings_use(
    X_train,
    model_use,
    batch_size=32
)

print(X_train_embeddings_use.shape)

Gerando embeddings USE:   0%|          | 0/471 [00:00<?, ?it/s]

(15050, 512)


### **6. Geração dos embeddings da validação**

In [34]:
X_val_embeddings_use = gerar_embeddings_use(
    X_val,
    model_use,
    batch_size=32
)

print(X_val_embeddings_use.shape)

Gerando embeddings USE:   0%|          | 0/118 [00:00<?, ?it/s]

(3763, 512)


### **7. Criação do pipeline**

In [35]:
pipeline_use_svm = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", SVC(class_weight="balanced"))
    ]
)

### **8. Treinamento do modelo**

In [36]:
pipeline_use_svm.fit(
    X_train_embeddings_use,
    y_train
)

Pipeline(steps=[('scaler', StandardScaler()),
                ('clf', SVC(class_weight='balanced'))])

### **9. Fazer as previsões**

In [37]:
y_pred_use_svm = pipeline_use_svm.predict(
    X_val_embeddings_use
)

accuracy_use_svm = accuracy_score(
    y_val,
    y_pred_use_svm
)

macro_f1_use_svm = f1_score(
    y_val,
    y_pred_use_svm,
    average="macro"
)

print(f"Accuracy: {accuracy_use_svm:.4f}")
print(f"Macro-F1: {macro_f1_use_svm:.4f}")
print(classification_report(y_val,y_pred_use_svm))

Accuracy: 0.4316
Macro-F1: 0.4294
              precision    recall  f1-score   support

          c1       0.43      0.46      0.45      1151
        c234       0.39      0.33      0.36      1292
          c5       0.47      0.50      0.48      1320

    accuracy                           0.43      3763
   macro avg       0.43      0.43      0.43      3763
weighted avg       0.43      0.43      0.43      3763



# **Testando variações**

### **1. Fazendo GridSearch - modelo SVM (NÃO FIZ)**

In [14]:
# from sklearn.svm import LinearSVC

# pipeline_use_linear = Pipeline(
#     steps=[
#         ("scaler", StandardScaler()),
#         ("clf", LinearSVC(
#             class_weight="balanced",
#             max_iter=10000
#         ))
#     ]
# )

In [15]:
# from sklearn.model_selection import GridSearchCV

# param_grid_use_svm = [
#     {
#         "clf__kernel": ["linear"],
#         "clf__C": [0.01, 0.1, 1, 10, 100]
#     },
#     {
#         "clf__kernel": ["rbf"],
#         "clf__C": [0.01, 0.1, 1, 10, 100],
#         "clf__gamma": ["scale", "auto"]
#     }
# ]

# # Configura o GridSearch
# grid_use_svm = GridSearchCV(
#     estimator=pipeline_use_svm,
#     param_grid=param_grid_use_svm,
#     scoring="accuracy",
#     cv=5,
#     n_jobs=-1,
#     return_train_score=True,
#     verbose=3
# )

# param_grid_use_linear = {
#     "clf__C": [0.001, 0.01, 0.1, 1, 10, 100]
# }

# grid_use_linear = GridSearchCV(
#     estimator=pipeline_use_linear,
#     param_grid=param_grid_use_linear,
#     scoring="accuracy",
#     cv=5,
#     n_jobs=-1,
#     return_train_score=True,
#     verbose=3
# )

### **1.1 Executa o grid**

In [16]:
# grid_use_svm.fit(
#     X_train_embeddings_use,
#     y_train
# )

# print("Melhores parâmetros:")
# print(grid_use_svm.best_params_)

# print("\nMelhor Accuracy média no CV:")
# print(grid_use_svm.best_score_)

# grid_use_linear.fit(
#     X_train_embeddings_use,
#     y_train
# )

# print("Melhores parâmetros:")
# print(grid_use_linear.best_params_)

# print("\nMelhor Accuracy média no CV:")
# print(grid_use_linear.best_score_)

In [17]:
# y_pred_use_grid = grid_use_svm.predict(
#     X_val_embeddings_use
# )

# accuracy_use_grid = accuracy_score(
#     y_val,
#     y_pred_use_grid
# )

# macro_f1_use_grid = f1_score(
#     y_val,
#     y_pred_use_grid,
#     average="macro"
# )

# print(f"Accuracy: {accuracy_use_grid:.4f}")
# print(f"Macro-F1: {macro_f1_use_grid:.4f}")

# print(
#     classification_report(
#         y_val,
#         y_pred_use_grid
#     )
# )

# y_pred_use_linear = grid_use_linear.predict(
#     X_val_embeddings_use
# )

# accuracy_use_linear = accuracy_score(
#     y_val,
#     y_pred_use_linear
# )

# macro_f1_use_linear = f1_score(
#     y_val,
#     y_pred_use_linear,
#     average="macro"
# )

# print(f"Accuracy: {accuracy_use_linear:.4f}")
# print(f"Macro-F1: {macro_f1_use_linear:.4f}")

# print("\nClassification Report:")
# print(
#     classification_report(
#         y_val,
#         y_pred_use_linear
#     )
# )

### **2. Validação Cruzada - variando seeds**
Conseiderando a configuração default do SVM (porque não fiz GridSearch)

In [20]:
from sklearn.model_selection import StratifiedKFold, cross_validate

seeds = [42, 100, 123]

for seed in seeds:

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=seed
    )

    results = cross_validate(
        pipeline_use_svm,
        X_train_embeddings_use,
        y_train,
        cv=cv,
        scoring={
            "accuracy": "accuracy",
            "f1_macro": "f1_macro"
        },
        n_jobs=-1
    )

    print(f"\nSeed: {seed}")
    print(
        f"Accuracy: {results['test_accuracy'].mean():.4f} "
        f"+/- {results['test_accuracy'].std():.4f}"
    )
    print(
        f"Macro-F1: {results['test_f1_macro'].mean():.4f} "
        f"+/- {results['test_f1_macro'].std():.4f}"
    )


Seed: 42
Accuracy: 0.4523 +/- 0.0080
Macro-F1: 0.4500 +/- 0.0076

Seed: 100
Accuracy: 0.4480 +/- 0.0081
Macro-F1: 0.4456 +/- 0.0077

Seed: 123
Accuracy: 0.4498 +/- 0.0080
Macro-F1: 0.4473 +/- 0.0080


In [ ]:
pipeline_use_svm.fit(
    X_train_embeddings_use,
    y_train
)

y_pred_use = pipeline_use_svm.predict(
    X_val_embeddings_use
)

accuracy_use = accuracy_score(
    y_val,
    y_pred_use
)

f1_use = f1_score(
    y_val,
    y_pred_use,
    average="macro"
)

print(f"Validation Accuracy: {accuracy_use:.4f}")
print(f"Validation Macro-F1: {f1_use:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred_use))